# CourtListener Copyright Case Tracker

Searches CourtListener for copyright opinions, downloads and indexes their text,
maintains a cumulative case list, and writes a markdown report.

**How to use:**
1. Edit the **Configuration** cell to adjust search terms and parameters.
2. Run all cells top-to-bottom: *Kernel → Restart & Run All*.
3. On subsequent runs you can re-run only the **Run Pipeline** cell at the bottom.

## 1. Setup — Imports & Logging

In [9]:
import os
import logging
import datetime
from zoneinfo import ZoneInfo
from dotenv import load_dotenv

from fetcher   import search_and_download
from indexer   import index_opinions, cleanup_db, cleanup_opinions
from differ    import save_snapshot
from reporter  import load_master, merge_into_master, save_master, write_report

_EASTERN = ZoneInfo("America/New_York")

def _eastern_time(timestamp):
    return datetime.datetime.fromtimestamp(timestamp, tz=_EASTERN).timetuple()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
)
for handler in logging.root.handlers:
    handler.formatter.converter = _eastern_time

logger = logging.getLogger(__name__)
print("Imports OK")

Imports OK


## 2. Configuration

Edit `ADDITIONAL_TERMS` to add or remove copyright-related keywords.
`"Copyright Act"` is always required — results must also contain at least one term from `ADDITIONAL_TERMS`.

In [10]:
# ── Search query terms ─────────────────────────────────────────────────────────
# "Copyright Act" is always required.
# At least one term from ADDITIONAL_TERMS must also appear.
# This prevents cases that only mention "infringement" or "authorship"
# in a non-copyright context from slipping through.

ADDITIONAL_TERMS = [
    "infringement", "fair use", "derivative work",
    "DMCA", "authorship", "originality",
]

def _build_query(additional):
    others = "(" + " OR ".join(f'"{t}"' for t in additional) + ")"
    return f'"Copyright Act" AND {others}'

QUERY = _build_query(ADDITIONAL_TERMS)
print("Query:\n", QUERY)

# ── Run parameters ──────────────────────────────────────────────────────────────
FILED_AFTER = "2019-01-01"  # Only cases filed after this date (YYYY-MM-DD)
MAX_RESULTS = 100           # Maximum cases to retrieve per run
ORDER_BY    = "score desc"  # Sort by relevance (v4 syntax)
COURT       = None          # Set to a court ID string to filter, or leave None

# ── Output directories / files ──────────────────────────────────────────────────
PDFS_DIR      = "pdfs"
DB_PATH       = "search_index.db"
SNAPSHOTS_DIR = "snapshots"
REPORTS_DIR   = "reports"
MASTER_PATH   = "master_cases.json"   # cumulative deduplicated case list

# ── API key ─────────────────────────────────────────────────────────────────────
load_dotenv()
API_KEY = os.getenv("COURTLISTENER_API_KEY")
if not API_KEY:
    print("WARNING: COURTLISTENER_API_KEY not found in .env — fetch will fail.")
else:
    print("API key loaded OK")

Query:
 "Copyright Act" AND ("infringement" OR "fair use" OR "derivative work" OR "DMCA" OR "authorship" OR "originality")
API key loaded OK


In [11]:
if not API_KEY:
    raise RuntimeError(
        "COURTLISTENER_API_KEY is not set. "
        "Add it to your .env file and re-run the Configuration cell."
    )

# 1 — Fetch cases and download opinions
results, errors = search_and_download(
    query=QUERY, filed_after=FILED_AFTER, order_by=ORDER_BY,
    max_results=MAX_RESULTS, court=COURT, api_key=API_KEY, pdfs_dir=PDFS_DIR,
)

# 2 — Index opinion text into SQLite; remove stale files and DB records
index_opinions(PDFS_DIR, DB_PATH, case_records=results)
current_ids = [r["cluster_id"] for r in results]
cleanup_db(current_ids, DB_PATH)
cleanup_opinions(current_ids, PDFS_DIR)

# 3 — Update the cumulative master case list
master = load_master(MASTER_PATH)
added  = merge_into_master(master, results)
save_master(master, MASTER_PATH)

# 4 — Save snapshot and write report (one file per day, overwrites if re-run same day)
save_snapshot(results, SNAPSHOTS_DIR)
report_path = write_report(
    master=master, added=added, query=QUERY, filed_after=FILED_AFTER,
    max_results=MAX_RESULTS, court=COURT, errors=errors, reports_dir=REPORTS_DIR,
)
print(f"\nDone! {len(master)} unique case(s) on record. Report: {report_path}")

2026-04-13 23:04:26,797 [INFO] Fetching page... (0 cases so far)
2026-04-13 23:04:28,555 [INFO] Fetching page... (20 cases so far)
2026-04-13 23:04:30,264 [INFO] Fetching page... (40 cases so far)
2026-04-13 23:04:32,030 [INFO] Fetching page... (60 cases so far)
2026-04-13 23:04:32,842 [INFO] Fetching page... (80 cases so far)
2026-04-13 23:04:33,612 [INFO] Retrieved 100 cases total.
2026-04-13 23:04:33,614 [INFO] Found 99 opinion file(s) to index.
2026-04-13 23:04:33,631 [INFO] Indexing complete. Indexed: 0, Skipped: 99
2026-04-13 23:04:33,636 [INFO] Master list saved: 130 unique case(s).
2026-04-13 23:04:33,638 [INFO] Snapshot saved to: snapshots/snapshot_2026-04-13.json
2026-04-13 23:04:33,639 [INFO] Report saved to: reports/report_2026-04-13.md



Done! 130 unique case(s) on record. Report: reports/report_2026-04-13.md
